# Welcome to EQ Lab

You're in **JupyterLab running on an EQ gateway** (served at `/jupyter`). This
environment ships with [`equser`](https://pypi.org/project/equser/) — the Python
toolkit for loading, analyzing, and visualizing continuous waveform (CPOW) and
power-monitor (PMon) data captured by EQ Wave sensors.

This notebook is a starting point: it checks your environment, shows where the
data lives, points you at the bundled tutorials, and runs one small example.

> These notebooks are copies. Edit freely — re-run `eq lab refresh` (or
> `equser notebooks refresh`) to restore the bundled originals.

## 1. Check your environment

Confirm `equser` is importable and list the notebooks bundled with it.

In [ ]:
import equser
from equser.notebooks import describe_notebooks

print(f"equser version: {equser.__version__}")
print("\nBundled notebooks:")
for nb in describe_notebooks():
    cat = nb["category"] or "(root)"
    print(f"  [{cat}] {nb['path']}")
    if nb.get("title"):
        print(f"        {nb['title']}")

## 2. Where the data lives

On a gateway, the captured Parquet files are under:

| Stream | Path | File naming |
|--------|------|-------------|
| CPOW (32 kHz waveform) | `/var/lib/eq-coherence/data/cpow/` | `YYYYMMDD_HHMMSS.parquet` |
| PMon (per-second summary) | `/var/lib/eq-coherence/data/pmon/` | `YYYYMMDD_HHMM.parquet` |

`equser.data` loads these with the raw ADC counts already scaled to volts/amps
using the `vscale`/`iscale` factors stored in each file's metadata.

In [ ]:
from pathlib import Path

cpow_dir = Path("/var/lib/eq-coherence/data/cpow")
pmon_dir = Path("/var/lib/eq-coherence/data/pmon")

for label, d in (("CPOW", cpow_dir), ("PMon", pmon_dir)):
    files = sorted(d.glob("*.parquet")) if d.exists() else []
    print(f"{label}: {len(files)} file(s) in {d}")
    if files:
        print(f"      most recent: {files[-1].name}")

## 3. Load a CPOW waveform

`load_cpow_scaled` returns a dict of NumPy arrays (one per channel: `VA`, `VB`,
`VC`, `IA`, `IB`, `IC`, `IN`) plus `start_time` and `sample_rate`. The cell below
loads the most recent CPOW file if one is present.

In [ ]:
from equser.data import load_cpow_scaled

cpow_files = sorted(cpow_dir.glob("*.parquet")) if cpow_dir.exists() else []
if cpow_files:
    result = load_cpow_scaled(str(cpow_files[-1]))
    print(f"Loaded {cpow_files[-1].name}")
    print(f"  start_time:  {result['start_time']}")
    print(f"  sample_rate: {result['sample_rate']} Hz")
    print(f"  VA peak:     {result['VA'].max():.1f} V")
else:
    print("No CPOW files yet. Connect a sensor, or work through")
    print("tutorials/01-parquet-files.ipynb with a sample file.")

## 4. Query this gateway's API

`equser.api.GatewayClient` (installed with the `[analysis]` extra, included in
`[jupyter]`) talks to the backend REST API. On the gateway itself, that's
`http://localhost:8080`.

In [ ]:
try:
    from equser.api import GatewayClient

    client = GatewayClient("http://localhost:8080")
    devices = client.list_devices()
    print(f"Devices on this gateway: {[d['id'] for d in devices]}")
except Exception as e:
    print(f"Gateway API not reachable from here ({e}).")
    print("That's fine — the data-loading cells above work offline.")

## Where to go next

- **`tutorials/01-parquet-files.ipynb`** — load and plot Parquet data directly
- **`tutorials/02-local-duckdb.ipynb`** — query files with SQL via DuckDB
- **`tutorials/03-backend-api.ipynb`** — REST + WebSocket access to a gateway
- **`tutorials/04-live-streaming.ipynb`** — stream live waveforms
- **`analysis/`** — harmonic analysis, power trends, delta analysis, AI event analysis

Full API reference: <https://pypi.org/project/equser/>.